<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Small_Regular_Graphs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graph Symmetry and Regular Structures

## Overview
This notebook explores the visualization of mathematical graphs that exhibit high degrees of symmetry, specifically focusing on Regular and Strongly Regular graphs. Using the Manim animation engine, these structures are rendered to highlight their combinatorial properties and real-world applications in network topology and quantum computing.

## Mathematical Concepts

### Regular Graphs
A graph is considered $k$-regular if every vertex has the same degree $k$. In other words, each node in the network is connected to exactly the same number of neighbors. Common examples include cycle graphs and complete graphs.

### Strongly Regular Graphs (SRGs)
A strongly regular graph with parameters $(n, k, \lambda, \mu)$ is a $k$-regular graph with $n$ vertices such that:
1. Every two adjacent vertices have exactly $\lambda$ common neighbors.
2. Every two non-adjacent vertices have exactly $\mu$ common neighbors.

These graphs are of significant interest in algebraic graph theory and finite geometry because of their highly constrained and symmetric nature.

## Visualization Goals
- Demonstrate the structural differences between 2D and 3D graph layouts.
- Provide context on the parameters that define the symmetry of each graph.
- Highlight practical applications of these abstract structures in fields such as distributed routing and molecular chemistry.

In [1]:
!sudo apt-get update
!sudo apt-get install -y libcairo2-dev libpango1.0-dev ffmpeg texlive texlive-latex-extra texlive-fonts-extra texlive-latex-recommended texlive-science texlive-fonts-recommended dvisvgm
!pip install manim

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 https://cli.github.com/packages stable InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,301 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,006 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,489 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
G

In [11]:
import manim
from manim.utils.ipython_magic import ManimMagic

try:
    # Manually register the %%manim magic command into the IPython shell
    get_ipython().register_magics(ManimMagic)
    print(f"Manim {manim.__version__} loaded and magic commands registered successfully.")
except Exception as e:
    print(f"Error loading Manim magic: {e}")

Manim 0.20.1 loaded and magic commands registered successfully.


In [35]:
%%manim -v WARNING -r 720,1280 RegularGraphExploration
"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
Concept: Regular and Strongly Regular Graph Visualization with Professional Markup
GitHub: github.com/zombimann/Mathematical-video-animations-and-visualization
"""

from manim import *
import numpy as np

# Explicitly set the internal camera frame to a vertical 9:16 aspect ratio
config.frame_width = 9
config.frame_height = 16

class RegularGraphExploration(ThreeDScene):
    # --- CONSTANTS ---
    GRAPH_SCALE = 1.6
    TITLE_SCALE = 0.8
    EDGE_WIDTH = 5
    NODE_RADIUS = 0.18

    # Timing
    REVEAL_NODES = 0.5
    REVEAL_EDGES = 1.0
    REVEAL_LABEL = 0.5
    HOLD_TIME = 3.5
    TRANSITION_PAUSE = 0.4

    def construct(self):
        self.camera.background_color = "#050505"

        # Permanent Watermark - Kept static on screen
        watermark = Text("© Mugambi Ndwiga / @craftsandengineering", font_size=16, fill_opacity=0.4)
        watermark.to_edge(DOWN, buff=0.3)
        self.add_fixed_in_frame_mobjects(watermark)

        # Header - Kept static on screen
        header = VGroup(
            Text("GRAPH SYMMETRY", weight=BOLD, font_size=40, color=BLUE_B),
            Line(LEFT, RIGHT, color=BLUE_E).scale(3)
        ).arrange(DOWN, buff=0.2).scale(self.TITLE_SCALE).to_edge(UP, buff=0.5)
        self.add_fixed_in_frame_mobjects(header)

        # Dataset structural definition: (Base Name, Math Markup, n, k, lam, mu, layout_type, category, use_case, is_3d)
        dataset = [
            ("Triangle", "<i>C</i><sub>3</sub>", 3, 2, 1, 0, "circular", "Strongly Regular", "Application: Mesh Generation & Graphics", False),
            ("Square", "<i>C</i><sub>4</sub>", 4, 2, 0, 2, "circular", "Strongly Regular", "Application: Distributed Network Routing", False),
            ("Tetrahedral", "<i>K</i><sub>4</sub>", 4, 3, 2, 0, "3d_tetra", "Strongly Regular", "Application: Quantum Computing & Molecular Bonds", True),
            ("Utility Graph", "<i>K</i><sub>3,3</sub>", 6, 3, 0, 3, "partite", "Strongly Regular", "Application: Printed Circuit Board (PCB) Layouts", False),
            ("Petersen Graph", "", 10, 3, 0, 1, "3d_petersen", "Strongly Regular", "Application: Network Topology Counter-Testing", True)
        ]

        for name, markup, n, k, lam, mu, l_type, cat, use_case, is_3d in dataset:
            self.play_graph_scene(name, markup, n, k, lam, mu, l_type, cat, use_case, is_3d)
            self.wait(self.TRANSITION_PAUSE)

        # Outro
        outro = VGroup(
            Text("Made by Mugambi Ndwiga", font_size=32, weight=BOLD),
            Text("@craftsandengineering", font_size=24, color=BLUE_C)
        ).arrange(DOWN, buff=0.4)

        self.add_fixed_in_frame_mobjects(outro)
        self.play(FadeOut(header), FadeIn(outro))
        self.wait(2.5)
        self.play(FadeOut(outro))

    def play_graph_scene(self, name, markup, n, k, lam, mu, l_type, cat, use_case, is_3d):
        # 1. Coordinate Space & Structural Setups
        if l_type == "circular":
            edges = [(i, (i+1)%n) for i in range(n)] if k==2 else [(i,j) for i in range(n) for j in range(i+1,n)]
            g = Graph(list(range(n)), edges, layout="circular")

        elif l_type == "partite":
            edges = [(i,j) for i in [0,1,2] for j in [3,4,5]]
            g = Graph(list(range(6)), edges, layout="partite", partitions=[[0,1,2],[3,4,5]])

        elif l_type == "3d_tetra":
            edges = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]
            pos_k4 = {
                0: np.array([0, 0, 1.1]),
                1: np.array([1.0, -0.57, -0.36]),
                2: np.array([-1.0, -0.57, -0.36]),
                3: np.array([0, 1.15, -0.36])
            }
            g = Graph(list(range(4)), edges, layout=pos_k4)

        elif l_type == "3d_petersen":
            edges = [(0,1),(1,2),(2,3),(3,4),(4,0), (5,7),(7,9),(9,6),(6,8),(8,5), (0,5),(1,6),(2,7),(3,8),(4,9)]
            pos_petersen = {}
            for i in range(5):
                angle = i * 2 * np.pi / 5 + np.pi/2
                pos_petersen[i] = np.array([1.4 * np.cos(angle), 1.4 * np.sin(angle), -0.4])
                pos_petersen[i+5] = np.array([0.75 * np.cos(angle), 0.75 * np.sin(angle), 0.4])
            g = Graph(list(range(10)), edges, layout=pos_petersen)

        # Styling adjustments & positioning graph upwards to clear the card panel safely
        g.set_color(BLUE_A).scale(self.GRAPH_SCALE).move_to(UP * 0.8)
        for v in g.vertices.values():
            v.scale_to_fit_width(self.NODE_RADIUS * 2)
        g.set_stroke(width=self.EDGE_WIDTH, opacity=0.85)

        # 2. UI Information Card Generation
        panel = RoundedRectangle(corner_radius=0.18, height=3.8, width=7.8, color=BLUE_E, stroke_width=1.5, fill_opacity=0.25)
        panel.to_edge(DOWN, buff=0.9)

        # Format title cleanly with bold and proper math-styled markup strings
        title_string = f"<b>{name} ({markup})</b>" if markup else f"<b>{name}</b>"
        title = MarkupText(title_string, font_size=32, color=YELLOW).next_to(panel.get_top(), DOWN, buff=0.35)

        # Format parameter variables (n, k) into elegant italics
        params_string = f"<i>n</i> = {n},  <i>k</i> = {k},  λ = {lam},  μ = {mu}"
        params = MarkupText(params_string, font_size=26, color=WHITE).next_to(title, DOWN, buff=0.25)

        cat_label = Text(cat, font_size=18, color=BLUE_B, weight=BOLD).next_to(params, DOWN, buff=0.25)
        use_case_label = Text(use_case, font_size=18, color=GREY_A, slant=ITALIC).next_to(cat_label, DOWN, buff=0.3)

        info_group = VGroup(panel, title, params, cat_label, use_case_label)
        self.add_fixed_in_frame_mobjects(info_group)
        info_group.set_opacity(0)

        # 3. Execution Sequences
        if is_3d:
            self.play(
                Create(VGroup(*g.vertices.values())),
                self.camera.theta_tracker.animate.set_value(-50 * DEGREES),
                self.camera.phi_tracker.animate.set_value(65 * DEGREES),
                run_time=self.REVEAL_NODES
            )
            self.play(Create(VGroup(*g.edges.values())), run_time=self.REVEAL_EDGES)
            self.play(info_group.animate.set_opacity(1).shift(UP * 0.1), run_time=self.REVEAL_LABEL)

            self.begin_ambient_camera_rotation(rate=0.22)
            self.wait(self.HOLD_TIME)
            self.stop_ambient_camera_rotation()

            self.play(
                FadeOut(g),
                FadeOut(info_group),
                self.camera.theta_tracker.animate.set_value(-90 * DEGREES),
                self.camera.phi_tracker.animate.set_value(0 * DEGREES),
                run_time=0.6
            )
        else:
            self.play(Create(VGroup(*g.vertices.values())), run_time=self.REVEAL_NODES)
            self.play(Create(VGroup(*g.edges.values())), run_time=self.REVEAL_EDGES)
            self.play(info_group.animate.set_opacity(1).shift(UP * 0.1), run_time=self.REVEAL_LABEL)
            self.wait(self.HOLD_TIME)
            self.play(FadeOut(g), FadeOut(info_group))

Manim Community v0.20.1